# VadCLIP Prompt Text-Space Scoring And Diversity Top-K Bank

Notebook này là **entry point chính để chọn prompt** (Hướng 1–2). Nó gọi `code/select_representative_class_prompts.py`, chấm điểm pool prompt trong text space CLIP/VadCLIP, rồi sinh:

- **Top-1** / class → ablation / diagnostics
- **Diversity Top-K + weights** → bank training mặc định

## Quality score (v2)

Các thành phần được **z-score theo class**, rồi gộp:

```text
quality =
  w_a * z(anchor)
+ w_m * z(anchor_margin)
+ w_c * z(centroid)
- w_h * z(hard_neg)
```

- `anchor`: gần tên class gốc
- `anchor_margin`: gần class mình hơn class khác
- `centroid`: gần mean embedding cùng class (thay pairwise `+intra`)
- `hard_neg`: xa prompt của class dễ lẫn (Abuse/Assault/Fighting, Arson/Explosion, …)

Sau đó chọn Top-K bằng **MMR / threshold** để tránh trùng lặp; softmax weight chỉ trên Top-K.

## Notebook liên quan

| Notebook | Vai trò |
|---|---|
| **This file** | Selection / scoring / build weighted bank (trước train) |
| `analyze_vadclip_class_prompt_alignment.ipynb` | Phân tích alignment sau fine-tune (theo 1 class, ví dụ Explosion) |

Mặc định: `raw_clip_encode_text` + `prototype_only` (encode đúng text prompt, không prepend class name khi chấm điểm).


In [ ]:
from pathlib import Path
import shlex
import subprocess
import sys

IN_COLAB = Path('/content').exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
else:
    NOTEBOOK_DIR = Path.cwd()
    PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR

SCRIPT_PATH = PROJECT_ROOT / 'code' / 'select_representative_class_prompts.py'
SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src'
MODEL_PATH = PROJECT_ROOT / 'model_ucf.pth'

if not SCRIPT_PATH.exists():
    raise FileNotFoundError(SCRIPT_PATH)
if not SRC_DIR.exists():
    raise FileNotFoundError(SRC_DIR)
if not MODEL_PATH.exists():
    raise FileNotFoundError(MODEL_PATH)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('SCRIPT_PATH:', SCRIPT_PATH)
print('MODEL_PATH:', MODEL_PATH)

## Install Dependencies

In [ ]:
if IN_COLAB:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', 'ftfy', 'regex', 'tqdm', 'scikit-learn', 'scipy', 'matplotlib'],
        check=True,
    )
print('Dependencies ready')

## Score Prompt Banks And Build Diversity Top-K Output

Cell này đọc pool prompt (ưu tiên short noun–verb mới), chạy quality score v2 + diversity Top-K, rồi ghi:

- `ucf_weighted_class_prompts.json` — **bank train chính** (Top-K + weights)
- `ucf_selected_representative_class_prompts.json` — Top-1 diagnostic / ablation

Có thể chạy thêm `legacy_auc8715` trong `SCORE_PRESETS_TO_RUN` để so với công thức cũ.


In [ ]:
# Primary pool: short noun-verb banks (new + merged).
# Optional: uncomment long_merged or old manual banks for broader candidate pools.
prompt_jsons = []
for pattern in [
    'ucf_class_prompts_short_nounverb_v1.json',
    'ucf_class_prompts_short_merged.json',
    # 'ucf_class_prompts_long_merged.json',
    # 'ucf_manual_class_prompts_5x*.json',
    # 'ucf_manual_class_prompts_10x*.json',
    # 'ucf_keyword_class_prompts_5x.json',
]:
    prompt_jsons.extend((PROJECT_ROOT / 'code').glob(pattern))
prompt_jsons = sorted({
    p.resolve() for p in prompt_jsons
    if 'selected' not in p.stem and 'representative' not in p.stem and 'weighted' not in p.stem
})

if not prompt_jsons:
    raise FileNotFoundError('No short prompt bank files found in code/')

# Put the newest noun-verb bank first (used as preferred source label in logs).
MAIN_SOURCE = (PROJECT_ROOT / 'code' / 'ucf_class_prompts_short_nounverb_v1.json').resolve()
if MAIN_SOURCE in prompt_jsons:
    prompt_jsons = [MAIN_SOURCE] + [p for p in prompt_jsons if p != MAIN_SOURCE]

print('Prompt JSON inputs:')
for p in prompt_jsons:
    marker = '  <-- preferred source' if p == prompt_jsons[0] else ''
    print(' -', p, marker)

BASE_OUTPUT_DIR = PROJECT_ROOT / 'Result' / 'vadclip_representative_prompt_selection'
SELECTED_OUTPUT_JSON = PROJECT_ROOT / 'code' / 'ucf_selected_representative_class_prompts.json'
WEIGHTED_OUTPUT_JSON = PROJECT_ROOT / 'code' / 'ucf_weighted_class_prompts.json'

# Quality presets (v2). balanced is the main training bank.
# Add 'legacy_auc8715' if you want a side-by-side with the old formula.
SCORE_PRESETS_TO_RUN = ['conservative', 'balanced', 'separation_focused']
MAIN_SCORE_PRESET = 'balanced'
INTER_MODE = 'max'  # max | top3_mean | mean
NORMALIZE_COMPONENTS = 'zscore'  # zscore | none
TOPK = 5
DIVERSITY_MODE = 'mmr'  # none | mmr | threshold
DIVERSITY_THRESHOLD = 0.97
MMR_LAMBDA = 0.70
MAIN_SPACE = 'raw_clip_encode_text'
SCORING_PROMPT_MODE = 'prototype_only'
PROMPT_WEIGHT_TEMPERATURE = 0.20
# For a faster local pass, use only MAIN_SPACE:
SPACES = ['raw_clip_encode_text']
# SPACES = ['vadclip_encode_textprompt', 'raw_clip_encode_text']

PRESET_OUTPUTS = {}
for score_preset in SCORE_PRESETS_TO_RUN:
    preset_output_dir = BASE_OUTPUT_DIR / score_preset
    if score_preset == MAIN_SCORE_PRESET:
        selected_output_json = SELECTED_OUTPUT_JSON
        weighted_output_json = WEIGHTED_OUTPUT_JSON
    else:
        selected_output_json = PROJECT_ROOT / 'code' / f'ucf_selected_representative_class_prompts_{score_preset}.json'
        weighted_output_json = PROJECT_ROOT / 'code' / f'ucf_weighted_class_prompts_{score_preset}.json'

    cmd = [
        sys.executable,
        str(SCRIPT_PATH),
        '--model-path', str(MODEL_PATH),
        '--output-dir', str(preset_output_dir),
        '--selected-output-json', str(selected_output_json),
        '--weighted-output-json', str(weighted_output_json),
        '--score-preset', score_preset,
        '--inter-mode', INTER_MODE,
        '--normalize-components', NORMALIZE_COMPONENTS,
        '--topk', str(TOPK),
        '--diversity-mode', DIVERSITY_MODE,
        '--diversity-threshold', str(DIVERSITY_THRESHOLD),
        '--mmr-lambda', str(MMR_LAMBDA),
        '--main-space', MAIN_SPACE,
        '--scoring-prompt-mode', SCORING_PROMPT_MODE,
        '--prompt-weight-temperature', str(PROMPT_WEIGHT_TEMPERATURE),
        '--spaces', *SPACES,
        '--prompt-jsons', *[str(p) for p in prompt_jsons],
    ]

    print('\n' + '=' * 100)
    print('Running score preset:', score_preset)
    print(' '.join(shlex.quote(part) for part in cmd))
    result = subprocess.run(cmd, check=False)
    if result.returncode != 0:
        raise RuntimeError(
            f'select_representative_class_prompts.py failed with exit code {result.returncode}. '
            f'See the script stdout/stderr printed above for the root cause.'
        )
    PRESET_OUTPUTS[score_preset] = {
        'output_dir': preset_output_dir,
        'selected_output_json': selected_output_json,
        'weighted_output_json': weighted_output_json,
    }

OUTPUT_DIR = PRESET_OUTPUTS[MAIN_SCORE_PRESET]['output_dir']
print('\nMain score preset:', MAIN_SCORE_PRESET)
print('Main selected (Top-1) JSON:', SELECTED_OUTPUT_JSON)
print('Main weighted (diversity Top-K) JSON:', WEIGHTED_OUTPUT_JSON)
print('Main output directory:', OUTPUT_DIR)
print('\nAll preset outputs:')
for preset, paths in PRESET_OUTPUTS.items():
    print(preset, '->', paths)


## Preview Top-1 And Diversity Top-K Weighted Bank

In Top-1 (ablation) và bank Top-K có weight (train). Kiểm tra thêm CSV `diversity_topk_prompts.csv` và `prompt_selection_scores_all_candidates.csv` trong `OUTPUT_DIR / MAIN_SPACE`.

Sau khi train multi-prompt với bank này, dùng `analyze_vadclip_class_prompt_alignment.ipynb` để xem class khó (ví dụ Explosion) align đúng prompt nào.


In [ ]:
import csv
import json

payload = json.loads(SELECTED_OUTPUT_JSON.read_text(encoding='utf-8'))
weighted_payload = json.loads(WEIGHTED_OUTPUT_JSON.read_text(encoding='utf-8'))
print('Selected prompt bank:', SELECTED_OUTPUT_JSON)
print('Selection method:', payload['metadata'].get('selection_method'))
print('Embedding space:', payload['metadata'].get('embedding_space'))
print('Scoring prompt mode:', payload['metadata'].get('scoring_prompt_mode'))
print('Normalize:', payload['metadata'].get('normalize_components'))
print('Score weights:', payload['metadata'].get('weights') or payload['metadata'].get('score_weights'))
print('\nTop-1 selected prompts:')
for class_name, prompts in payload['classes'].items():
    detail = payload.get('selection_details', {}).get(class_name, {})
    print(
        f"{class_name:<13} score={detail.get('selection_score', 0):+.4f} "
        f"anchor={detail.get('anchor_similarity', 0):.4f} "
        f"centroid={detail.get('centroid_similarity', 0):.4f} "
        f"hard_neg={detail.get('hard_negative_prompt_similarity', 0):.4f} "
        f"margin={detail.get('anchor_margin', 0):+.4f} "
        f"source={detail.get('source_stem', '')}#{detail.get('prompt_index', '')} | {prompts[0]}"
    )

meta_w = weighted_payload.get('metadata', {})
print('\nWeighted / diversity Top-K bank:', WEIGHTED_OUTPUT_JSON)
print('  topk:', meta_w.get('topk'), '| diversity:', meta_w.get('diversity_mode'),
      '| tau:', meta_w.get('diversity_threshold'), '| mmr_lambda:', meta_w.get('mmr_lambda'))
print('  prompts_per_class:', meta_w.get('prompts_per_class'))
print('  selection_method:', meta_w.get('selection_method'))
for class_name, prompts in weighted_payload['classes'].items():
    weights = weighted_payload['prompt_weights'][class_name]
    pairs = sorted(zip(weights, prompts), reverse=True)
    print(f'\n{class_name}:')
    for weight, prompt in pairs:
        print(f'  weight={weight:.4f} | {prompt}')

div_csv = OUTPUT_DIR / MAIN_SPACE / 'diversity_topk_prompts.csv'
if div_csv.exists():
    print('\nDiversity Top-K CSV:', div_csv)
    with div_csv.open('r', encoding='utf-8', newline='') as f:
        rows = list(csv.DictReader(f))
    print('Rows:', len(rows))

sim_csv = OUTPUT_DIR / MAIN_SPACE / 'prompt_to_class_anchor_similarity.csv'
print('\nLowest prompt-to-own-class similarity rows:')
with sim_csv.open('r', encoding='utf-8', newline='') as f:
    rows = list(csv.DictReader(f))
rows = sorted(rows, key=lambda r: float(r['own_class_similarity']))
for row in rows[:30]:
    print(
        f"{row['class_name']:<13} {row['source_stem']}#{row['prompt_index']:<2} "
        f"own={float(row['own_class_similarity']):.4f} "
        f"best_other={row['best_other_anchor_class']}:{float(row['best_other_anchor_similarity']):.4f} "
        f"margin={float(row['anchor_margin']):+.4f} | {row['raw_prompt_text']}"
    )
